# Cafe Sales Data Cleaning
This notebook demonstrates a complete data cleaning and preprocessing workflow for the dirty cafe sales dataset using Python, Pandas, and NumPy.

In [67]:
import pandas as pd
import numpy as np

## Load Dataset

In [84]:
df = pd.read_csv('dirty_cafe_sales.csv')
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,UNKNOWN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02


In [85]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


## Handle invalid and missing values
Replace ERROR and UNKNOWN with NaN.

In [86]:
invalid_values = ['ERROR','UNKNOWN','']
df = df.replace(invalid_values, np.nan)

## Fix data types
Convert numeric and date columns.

In [87]:
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["Price Per Unit"] = pd.to_numeric(df["Price Per Unit"], errors="coerce")
df["Total Spent"] = pd.to_numeric(df["Total Spent"], errors="coerce")

df["Transaction Date"] = pd.to_datetime(df["Transaction Date"], errors="coerce")

## Fill missing numeric values
Use median.

In [ ]:
df["Quantity"].fillna(df["Quantity"].median(), inplace=True)
df["Price Per Unit"].fillna(df["Price Per Unit"].median(), inplace=True)

## Fix item names
For items with the same price, use the most frequent item in the dataset.

In [89]:
mapping = (
    df.dropna(subset=["Item", "Price Per Unit"])
    .groupby("Price Per Unit")["Item"]
    .agg(lambda x: x.value_counts().idxmax())
)

df["Item"] = df.apply(
    lambda r: mapping[r["Price Per Unit"]] 
    if pd.isna(r["Item"]) and r["Price Per Unit"] in mapping 
    else r["Item"], axis=1
)



## Fix total and price relationship
Total = Quantity × Price

In [ ]:
df["Total Spent"].fillna(df["Quantity"] * df["Price Per Unit"], inplace=True)

In [92]:
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,NaN,NaN,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2.0,2.0,4.0,NaN,NaN,2023-08-30
9996,TXN_9659401,Juice,3.0,3.0,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4.0,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3.0,3.0,3.0,Digital Wallet,NaN,2023-12-02


## Handling Missing Values and Improving Data Quality
### Payment Method
### Location
### Transaction Date

In [ ]:
df["Payment Method"].fillna(df["Payment Method"].mode()[0], inplace=True)
df["Location"].fillna("Unknown", inplace=True)
df.dropna(subset=["Transaction Date"], inplace=True)

In [94]:
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,Digital Wallet,Unknown,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2.0,2.0,4.0,Digital Wallet,Unknown,2023-08-30
9996,TXN_9659401,Juice,3.0,3.0,3.0,Digital Wallet,Unknown,2023-06-02
9997,TXN_5255387,Coffee,4.0,2.0,8.0,Digital Wallet,Unknown,2023-03-02
9998,TXN_7695629,Cookie,3.0,3.0,3.0,Digital Wallet,Unknown,2023-12-02


## Create season column

In [95]:
def Season(d):
    if pd.isna(d): return np.nan
    if d.month in [12,1,2]: return "Winter"
    if d.month in [3,4,5]: return "Spring"
    if d.month in [6,7,8]: return "Summer"
    return "Fall"

df["Season"] = df["Transaction Date"].apply(Season)

## Data Type Optimization

In this step, we improved the data types to better reflect real-world business logic and enhance performance.

### Quantity
The **Quantity** column was converted from float to integer because quantity represents discrete values in transactions. This ensures data accuracy and avoids unnecessary decimal values.

### Categorical Features
Several columns such as **Item, Payment Method, Location, and Season** were converted to the **category** data type.  
This optimization helps:
- Reduce memory usage.
- Improve performance in large datasets.
- Better represent categorical business features.

In [96]:
df["Quantity"] = df["Quantity"].astype(int)
df["Item"] = df["Item"].astype("category")
df["Payment Method"] = df["Payment Method"].astype("category")
df["Location"] = df["Location"].astype("category")
df["Season"] = df["Season"].astype("category")

In [97]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9540 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    9540 non-null   object        
 1   Item              9540 non-null   category      
 2   Quantity          9540 non-null   int64         
 3   Price Per Unit    9540 non-null   float64       
 4   Total Spent       9540 non-null   float64       
 5   Payment Method    9540 non-null   category      
 6   Location          9540 non-null   category      
 7   Transaction Date  9540 non-null   datetime64[ns]
 8   Season            9540 non-null   category      
dtypes: category(4), datetime64[ns](1), float64(2), int64(1), object(1)
memory usage: 485.3+ KB


In [99]:
df.isna().sum()

Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
Season              0
dtype: int64

In [100]:
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,Season
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08,Fall
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16,Spring
2,TXN_4271903,Cookie,4,1.0,4.0,Credit Card,In-store,2023-07-19,Summer
3,TXN_7034554,Salad,2,5.0,10.0,Digital Wallet,Unknown,2023-04-27,Spring
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11,Summer
...,...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,Digital Wallet,Unknown,2023-08-30,Summer
9996,TXN_9659401,Juice,3,3.0,3.0,Digital Wallet,Unknown,2023-06-02,Summer
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,Unknown,2023-03-02,Spring
9998,TXN_7695629,Cookie,3,3.0,3.0,Digital Wallet,Unknown,2023-12-02,Winter


## Save cleaned dataset

In [101]:
df.to_csv('cleaned_cafe_sales.csv', index=False)